# ABBA4 outer-projection configuration comparison

This experiment compares four configurations of `ABBA4Implicit` in the measured
GC2D `PHI_2.h5` potential: reduced or simultaneous spatial projection, each solved
with Newton or Broyden. One projection surrounds the complete composition.
Passive energy tracking adds time and momentum to the accepted workspace (R6 per
particle); the nonlinear solve contains 2 or 6 spatial unknowns per particle.

The physical parameters, initial positions, reference settings, time interval,
and step size remain explicit below. Each of the four configurations advances
`PARTICLE_COUNT` independent trajectories.

This notebook supersedes the former sixteen-configuration R6/R8 comparison.
Full time/momentum projection and projection after every ABBA factor are retired.
Previously stored outputs were cleared because they describe different methods;
the versioned history retains those historical results. New results must not be
interpreted as a rerun of the former R8 experiment.


In [ ]:
from os import cpu_count
from pathlib import Path
import hashlib
from time import perf_counter

import h5py
import numpy as np

from diagnostics import (
    load_reference_trajectory,
    reference_trajectory_output_directory,
)
from diagnostics.paths import find_project_root
from potential import Potential, load_gc2d_h5_potential
from studies import (
    ABBA4ConfigurationComparisonConfig,
    HighPrecisionReferenceConfig,
    latin_hypercube_gc_configuration,
    run_abba4_configuration_comparison,
    run_high_precision_reference_trajectory,
)
from visualization import (
    animate_abba4_configuration_trajectories,
    display_animation,
    display_records_table,
)

## Reproducible configuration

`FULL_STUDY=True` selects the scientific experiment on $[0,4]$. For a
short execution check, change only that flag to `False`; the quick profile
uses $[0,0.05]$ while retaining the real HDF5 potential, the same three
initial conditions, the same integration step and tolerances, and the
matching prefix of the audited reference.

The ignored reference directory is reused when present. If it is absent, the
notebook regenerates it reproducibly with the original DOP853 solve and an
independent Radau audit over the complete interval $[0,4]$. This bootstrap can
take several minutes even in quick mode, but it happens only once. Potential
loading, reference loading or generation, and animation rendering are outside
every per-configuration runtime. `WORKER_COUNT` is an explicit execution
control: complete one-particle trajectories run in separate spawned processes,
while each trajectory's time steps remain sequential. Each worker is limited to
one native numerical-library thread, preventing nested oversubscription. With
`PROGRESS=True`, the parent process prints an immediate global progress bar
and a heartbeat approximately every five seconds, including completed and
unfinished trajectories, worker count, elapsed time, and ETA. Each completion
also identifies its configuration and particle. Per-step progress bars are
available only when `WORKER_COUNT=1`. Keep
`WORKER_COUNT` at least one below the logical CPU count when moving this
notebook to another workstation or cloud VM.

In [ ]:
FULL_STUDY = True

NOTEBOOK_PATH = Path(
    "notebooks/experiments/comparison/"
    "compare_abba4_r6_r8_configurations.ipynb"
)
PROJECT_ROOT = find_project_root(Path.cwd())
POTENTIAL_RELATIVE_PATH = Path("data/potential/V1/PHI_2.h5")
POTENTIAL_PATH = PROJECT_ROOT / POTENTIAL_RELATIVE_PATH
REFERENCE_SOURCE_NOTEBOOK_PATH = NOTEBOOK_PATH
REFERENCE_NAME = "phi_2_heterogeneous_3"
REFERENCE_VERSION = "v2"
REFERENCE_DIRECTORY = reference_trajectory_output_directory(
    NOTEBOOK_PATH,
    reference_name=REFERENCE_NAME,
    version=REFERENCE_VERSION,
    project_root=PROJECT_ROOT,
)

B = 1.5
FIELD_INDICES = (0, 1)
GRID_NX = 128
GRID_NY = 128
DENOISING = False
DENOISING_SIGMA = 1.0
INTERPOLATION_ORDER = 3

PARTICLE_COUNT = 3
INITIAL_CONDITION_SEED = 20260827
DOMAIN_MARGIN_FRACTION = 0.05
RHO = 0.0

REFERENCE_T_SPAN = (0.0, 4.0)
REFERENCE_SAVE_INTERVAL = 0.01
DOP853_RELATIVE_TOLERANCE = 1e-12
DOP853_ABSOLUTE_TOLERANCE = 1e-14
DOP853_STEPS_PER_FIELD_PERIOD = 8
RADAU_RELATIVE_TOLERANCE = 1e-12
RADAU_ABSOLUTE_TOLERANCE = 1e-14
RADAU_STEPS_PER_FIELD_PERIOD = 16

T_SPAN = (0.0, 4.0 if FULL_STUDY else 0.05)
INTEGRATION_STEP = 0.0025
SAVE_INTERVAL = 0.01
ABSOLUTE_TOLERANCE = 1e-14
RELATIVE_TOLERANCE = 1e-13
MAX_ITERATIONS = 40
AVAILABLE_LOGICAL_CPU_COUNT = cpu_count() or 1
# Use the 32-vCPU cloud host while reserving one logical CPU for the system.
WORKER_COUNT = min(31, max(1, AVAILABLE_LOGICAL_CPU_COUNT - 1))
if WORKER_COUNT > max(1, AVAILABLE_LOGICAL_CPU_COUNT - 1):
    raise ValueError(
        "WORKER_COUNT must leave one logical CPU for the system on this host."
    )
PROGRESS = True

print(f"Notebook: {NOTEBOOK_PATH}")
print(f"Study mode: {'full' if FULL_STUDY else 'quick validation'}")
print(f"Live trajectory progress: {'enabled' if PROGRESS else 'disabled'}")

## Real potential, audited reference, and initial conditions

The source file is a Git LFS asset. The HDF5 check below distinguishes the
real 600 MB field from an unresolved LFS pointer. Its checksum, selected
mode, interpolation grid, and deterministic Latin-hypercube initial state
define the complete potential and initial-condition metadata.

If the ignored reference artifact is missing, the public reference-study API
recreates `outputs/experiments/comparison/phi_2_heterogeneous_3/v2` for the nondimensional
runtime potential. DOP853 uses at least eight maximum steps per field
period and Radau at least sixteen; each maximum step is also limited by the
saved-time interval. Both use relative tolerance $10^{-12}$ and absolute
tolerance $10^{-14}$. The regenerated
artifact retains the Euclidean distance convention required by this
non-periodic HDF5 field.

In [ ]:
if not POTENTIAL_PATH.is_file() or not h5py.is_hdf5(POTENTIAL_PATH):
    raise FileNotFoundError(
        "The real PHI_2 HDF5 asset is unavailable. Fetch the Git LFS file at "
        f"{POTENTIAL_PATH}."
    )

with POTENTIAL_PATH.open("rb") as stream:
    potential_source_sha256 = hashlib.file_digest(stream, "sha256").hexdigest()

potential = load_gc2d_h5_potential(
    POTENTIAL_PATH,
    B=B,
    indx=FIELD_INDICES,
    nx=GRID_NX,
    ny=GRID_NY,
    denoising=DENOISING,
    sigma=DENOISING_SIGMA,
    interpolation_order=INTERPOLATION_ORDER,
)
initial_configuration = latin_hypercube_gc_configuration(
    potential,
    particle_count=PARTICLE_COUNT,
    seed=INITIAL_CONDITION_SEED,
    domain_margin_fraction=DOMAIN_MARGIN_FRACTION,
)

grid = potential.grid
domain_min = np.asarray((grid.xmin, grid.ymin))
domain_max = np.asarray((grid.xmax, grid.ymax))
domain_span = domain_max - domain_min
potential_metadata = {
    "format": "GC2D_HDF5",
    "source_path": str(POTENTIAL_RELATIVE_PATH),
    "source_sha256": potential_source_sha256,
    "B": B,
    "field_indices": FIELD_INDICES,
    "nx": GRID_NX,
    "ny": GRID_NY,
    "denoising": DENOISING,
    "denoising_sigma": DENOISING_SIGMA,
    "interpolation_order": INTERPOLATION_ORDER,
    "selected_source_field_indices": potential.metadata.source_field_indices,
    "selected_frequencies": potential.frequencies,
    "selected_source_frequencies": potential.metadata.source_frequencies,
    "characteristic_length": potential.metadata.characteristic_length,
    "characteristic_period": potential.metadata.characteristic_period,
    "normalization_factor": potential.metadata.normalization_factor,
}
initial_condition_metadata = {
    "particle_count": PARTICLE_COUNT,
    "seed": INITIAL_CONDITION_SEED,
    "domain_margin_fraction": DOMAIN_MARGIN_FRACTION,
    "sampling_min": domain_min + DOMAIN_MARGIN_FRACTION * domain_span,
    "sampling_max": domain_max - DOMAIN_MARGIN_FRACTION * domain_span,
    "sampling": "seeded_latin_hypercube_independent_axis_jitter",
}

field_frequency = float(np.min(np.abs(potential.frequencies)))
field_period = 2.0 * np.pi / field_frequency
dop853_maximum_step = min(
    REFERENCE_SAVE_INTERVAL,
    field_period / DOP853_STEPS_PER_FIELD_PERIOD,
)
radau_maximum_step = min(
    dop853_maximum_step,
    field_period / RADAU_STEPS_PER_FIELD_PERIOD,
)
reference_config = HighPrecisionReferenceConfig(
    t_span=REFERENCE_T_SPAN,
    save_interval=REFERENCE_SAVE_INTERVAL,
    rho=RHO,
    distance_convention="euclidean",
    relative_tolerance=DOP853_RELATIVE_TOLERANCE,
    absolute_tolerance=DOP853_ABSOLUTE_TOLERANCE,
    maximum_step=dop853_maximum_step,
    audit_relative_tolerance=RADAU_RELATIVE_TOLERANCE,
    audit_absolute_tolerance=RADAU_ABSOLUTE_TOLERANCE,
    audit_maximum_step=radau_maximum_step,
)

if REFERENCE_DIRECTORY.is_dir():
    reference = load_reference_trajectory(REFERENCE_DIRECTORY)
    reference_origin = "loaded existing artifact"
else:
    reference_result = run_high_precision_reference_trajectory(
        potential,
        initial_configuration,
        notebook_path=REFERENCE_SOURCE_NOTEBOOK_PATH,
        config=reference_config,
        potential_metadata=potential_metadata,
        initial_condition_metadata=initial_condition_metadata,
        reference_name=REFERENCE_NAME,
        version=REFERENCE_VERSION,
        project_root=PROJECT_ROOT,
        overwrite=False,
    )
    reference = reference_result.trajectory
    reference_origin = "regenerated with DOP853 and Radau"

assert isinstance(potential, Potential)
assert reference.paths.directory.resolve() == REFERENCE_DIRECTORY.resolve()
assert reference.metadata["potential"]["source_sha256"] == potential_source_sha256
assert reference.metadata["config"]["distance_convention"] == "euclidean"
assert tuple(reference.metadata["config"]["t_span"]) == REFERENCE_T_SPAN
assert reference.metadata["config"]["save_interval"] == REFERENCE_SAVE_INTERVAL
assert reference.metadata["config"]["relative_tolerance"] == DOP853_RELATIVE_TOLERANCE
assert reference.metadata["config"]["absolute_tolerance"] == DOP853_ABSOLUTE_TOLERANCE
assert reference.metadata["config"]["audit_relative_tolerance"] == RADAU_RELATIVE_TOLERANCE
assert reference.metadata["config"]["audit_absolute_tolerance"] == RADAU_ABSOLUTE_TOLERANCE
assert reference.metadata["reference_name"] == REFERENCE_NAME
assert reference.metadata["particle_count"] == PARTICLE_COUNT
assert np.array_equal(initial_configuration.initial_state, reference.initial_state)

print(f"Potential: {POTENTIAL_RELATIVE_PATH}")
print(f"SHA-256: {potential_source_sha256}")
print(f"Selected source fields: {potential.metadata.source_field_indices.tolist()}")
print(f"Selected runtime frequencies: {potential.frequencies.tolist()}")
print(f"Selected source frequencies: {potential.metadata.source_frequencies.tolist()}")
print(f"Interpolated grid: {potential.grid.shape}")
print(f"Reference: {reference.paths.directory} ({reference_origin})")
print(f"Reference DOP853 maximum step: {reference_config.maximum_step:.16e}")
print(f"Reference Radau maximum step: {reference_config.audit_maximum_step:.16e}")
print(f"Reference audit global RMS: {reference.metadata['audit']['global_rms_distance']:.9e}")

## Four configurations

The study dispatches four configurations times `PARTICLE_COUNT` complete
trajectories. Newton tasks are scheduled first. Every run uses the same physical
initial states, output times, tolerances, and reference samples. Runtime excludes
potential loading and reference generation.


In [ ]:
comparison_config = ABBA4ConfigurationComparisonConfig(
    t_span=T_SPAN,
    particle_count=PARTICLE_COUNT,
    integration_step=INTEGRATION_STEP,
    save_interval=SAVE_INTERVAL,
    rho=RHO,
    absolute_tolerance=ABSOLUTE_TOLERANCE,
    relative_tolerance=RELATIVE_TOLERANCE,
    max_iterations=MAX_ITERATIONS,
    progress=PROGRESS,
    worker_count=WORKER_COUNT,
)

assert comparison_config.particle_count == PARTICLE_COUNT
assert comparison_config.worker_count == WORKER_COUNT

print(comparison_config)
print(f"Complete steps per trajectory: {comparison_config.step_count}")
print(f"Saved samples per trajectory: {comparison_config.output_sample_count}")
print(f"Worker processes: {comparison_config.worker_count}")
print(f"Available logical CPUs: {AVAILABLE_LOGICAL_CPU_COUNT}")

In [ ]:
study_started = perf_counter()
result = run_abba4_configuration_comparison(
    potential,
    initial_configuration,
    reference,
    config=comparison_config,
)
study_wall_seconds = perf_counter() - study_started
print(
    f"Parallel study wall time: {study_wall_seconds:.1f} s "
    f"with {comparison_config.worker_count} worker processes."
)

## Contract checks

Assert four ordered variants, `PARTICLE_COUNT` trajectories per variant, aligned
saved times, and five numerical metrics per row.


In [ ]:
summary_rows = result.summaries()
metric_fields = (
    "mean_trajectory_error",
    "final_trajectory_error",
    "total_runtime_seconds",
    "mean_iterations_per_solve",
    "mean_relative_energy_error",
)

assert len(result.variants) == 4
assert len(result.solutions) == 4
assert len(summary_rows) == 4
assert len(metric_fields) == 5
assert tuple(row.key for row in summary_rows) == tuple(
    variant.key for variant in result.variants
)
assert all(
    len(result.solutions[variant.key]) == PARTICLE_COUNT
    for variant in result.variants
)
total_trajectory_count = sum(
    len(result.solutions[variant.key]) for variant in result.variants
)
assert total_trajectory_count == len(result.variants) * PARTICLE_COUNT

for variant in result.variants:
    for solution in result.solutions[variant.key]:
        assert solution.states.shape == (2, result.times.size)
        assert np.array_equal(solution.t, result.times)

metric_values = np.asarray(
    [[getattr(row, field) for field in metric_fields] for row in summary_rows],
    dtype=float,
)
assert metric_values.shape == (4, 5)
assert np.all(np.isfinite(metric_values))
assert np.all(metric_values >= 0.0)
assert comparison_config.worker_count == WORKER_COUNT
assert np.isfinite(study_wall_seconds) and study_wall_seconds > 0.0

print(f"Configurations: {len(result.variants)}")
print(f"Independent trajectories: {total_trajectory_count}")
print(f"Metric matrix shape: {metric_values.shape}")
print(f"Study wall time: {study_wall_seconds:.1f} s")

## Comparison table: four configurations and five metrics

Each row reports trajectory RMS error, final trajectory error, summed runtime,
mean nonlinear corrections per solve, and mean relative generalized-energy error.
The generalized energy is physical H plus passive momentum kappa. Under parallel
execution the runtime includes CPU contention; use one worker for timing studies.


In [ ]:
display_records_table(
    summary_rows,
    columns=(
        ("label", "Configuration (row index)", None),
        ("mean_trajectory_error", "Mean trajectory RMS error", ".9e"),
        ("final_trajectory_error", "Final RMS error", ".9e"),
        ("total_runtime_seconds", "Aggregate worker runtime [s]", ".6f"),
        ("mean_iterations_per_solve", "Mean iterations / solve", ".6f"),
        (
            "mean_relative_energy_error",
            "Mean relative generalized-energy error",
            ".9e",
        ),
    ),
)

## Synchronized trajectory animation

The four panels show reduced/Newton, reduced/Broyden, simultaneous/Newton, and
simultaneous/Broyden configurations. Every panel shares the same initial-position
colors, physical field, and saved times. The animation is a downsampled view.


In [ ]:
animation_frame_count = (
    101 if FULL_STUDY else min(21, int(result.times.size))
)
animation = animate_abba4_configuration_trajectories(
    result,
    frames=animation_frame_count,
    interval=100,
    repeat=True,
)
display_animation(animation, embed_limit_mb=100.0)

## Interpretation notes

The full-study table is a controlled comparison on one potential, one set of
initial conditions, and one fixed integration grid. With parallel execution,
the runtime column sums task wall times measured under CPU contention; it is
useful operational context but not an isolated method benchmark or the campaign
makespan. Use `WORKER_COUNT=1` for fair serial runtime comparisons. Newton and
Broyden iteration counts still measure deterministic nonlinear corrections,
while their per-iteration costs can differ.

Generalized energy, rather than the time-dependent physical Hamiltonian alone,
is the conserved extended quantity. The quick profile is only an execution
and contract check; scientific conclusions must use `FULL_STUDY=True`.